# NB-14 — Replicate a ladder rung at a new seed

**This is NB-01 with two lines changed.** Every other cell is byte-identical, deliberately:
a replicate is only a replicate if the recipe is untouched.

## What this measures

The paper's headline (`sec:replication`) is that two runs of the 1-hour rung, launched from an
identical configuration **at the same seed 1337**, diverged under fp16 + cuDNN autotune and
ended 34 pp apart in targeted attack success. That is one pair, at one rung, from runs that
were not *intended* to differ.

A reviewer's first question is therefore: *how do you know 34 pp is typical?* You do not — two
runs give a difference, not a spread. This notebook produces the runs that give a spread:
deliberate replicates at a **declared** seed, so the between-run standard deviation can be
estimated per rung and pooled.

## What must NOT change

`MAX_STEPS`, `EARLY_STOP_PATIENCE`, `BATCH`, `GRAD_ACCUM`, `LR`, `WARMUP`, `MAX_DUR`,
`BASE_MODEL`, `USE_BOTH_GPUS`, and the `RUNG >= 25` milestone gate. A guard cell asserts all
of them against the published ladder's values and refuses to start if any differs. Changing
one would make the new model a different condition rather than a second draw from the same one.

**`cudnn.benchmark` stays `True` and fp16 stays on.** These are the nondeterminism sources the
paper measures. Turning them off would answer a question nobody asked.

## Run order — TWO runs per rung, not one

One commit per `(RUNG, SEED)`. **Six runs:**

| commit | `RUNG` | `SEED` | publishes |
|---|---|---|---|
| 1 | `1`  | `1338` | `ckpt-xlsr-1h-s1338` |
| 2 | `1`  | `1339` | `ckpt-xlsr-1h-s1339` |
| 3 | `5`  | `1338` | `ckpt-xlsr-5h-s1338` |
| 4 | `5`  | `1339` | `ckpt-xlsr-5h-s1339` |
| 5 | `25` | `1338` | `ckpt-xlsr-25h-s1338` |
| 6 | `25` | `1339` | `ckpt-xlsr-25h-s1339` |

~3.5 GPU-h each (6000 steps at ~2.1 s/step; early stopping may cut it shorter).
Two can run concurrently.

### Why two per rung and not one

Kaggle now ships `transformers` **5.16.1**; the published ladder trained on **4.44.2**. So a
replicate compared against its published sibling differs in *two* things — the run and the
library version — and a reviewer would be right to say the resulting sigma conflates them.

A **pair of new runs at the same rung** has no such problem: both on 5.16.1, only the run
differs. Two runs per rung gives three clean within-environment pairs at three exposures
(pooled df 3) instead of three confounded cross-version comparisons.

The extra cost is two trainings and two attacks, about 16 GPU-h. It buys the only version of
this measurement that is not open to that objection.

## Attach

`bn-asr-splits`, `bengaliai-speech` (Competitions -> "bengaliai speech", rules accepted).
GPU T4 x2.

## Internet: ON for this notebook only

**Settings → Internet → On.** This notebook fine-tunes from
`facebook/wav2vec2-xls-r-300m`, a ~1.2 GB Hugging Face Hub download that is neither on
Kaggle's disk nor in any attached dataset. With Internet off there is no way to get it: DNS
fails and `huggingface_hub` retries with exponential backoff across three files — **840 s
before the notebook raises, with nothing trained.** A pre-flight check in the configuration
cell now catches that in about five seconds and tells you to flip the switch.

**NB-15, NB-16 and NB-17 are the opposite: Internet off.** They load models from attached
Kaggle datasets by local path and make no hub call at all.

Turning Internet on is safe here because **there is no pip install in this notebook.** NB-01's
pip cell is gone — it installed `jiwer`, which is not preinstalled and which `compute_metrics`
calls at every eval step. WER and CER are now computed inline with jiwer's exact list
semantics, self-tested in the same cell. So the network is used for exactly one thing, the base
model, and cannot silently re-pin a package version.

**The environment caveat.** Without the pin, this trains on whatever Kaggle currently
preinstalls rather than the `transformers` 4.44.2 / `accelerate` 1.13.0 the published ladder
used. So:

- **`s1338` vs `s1339` at the same rung is the clean comparison** — one environment, only the
  run differs. Run the replicates close together in time to keep it that way.
- **replicate vs published** carries a package-version difference on top of the run difference.
  Report it; do not pretend it is not there.

Attack-side ASR is unaffected: NB-16 attacks every checkpoint in one session on one version.

**Publish each output as its own dataset.** Never as a new version of `ckpt-xlsr-{RUNG}h` —
that would replace the weights every published number was measured on.


## 1 · Configuration — change `RUNG` and `SEED` for each run

In [ ]:
# ---------------------------------------------------------------------------
# RUNG = HOURS OF TRAINING AUDIO. Not hours of runtime. See the notes above.
# ---------------------------------------------------------------------------
RUNG = 1          # <<<<<< which rung to replicate: 1, 5 or 25 (hours of AUDIO)
SEED = 1338       # <<<<<< the ONLY thing that may differ from the published run.
                  #        1337 is the published ladder. Use 1338, 1339, ...
                  #
                  # Read SEED as a label, not a guarantee. fp16 and cuDNN autotune
                  # make this training nondeterministic even at a FIXED seed -- that
                  # is the paper's finding, not a defect. Declaring a new seed makes
                  # each run an intentional draw and keeps the dataset names honest.

# The ladder has already been trained four times on this exact configuration, so
# the stack is proven and the probe is 10 min of quota per replicate for no new
# information. Default False.
#
# Flip it True for ONE commit if Kaggle's base image has changed under you (a new
# torch or transformers in the version banner) -- then the probe is worth it again.
# Input resolution and audio-path checks happen in section 3 regardless, within
# seconds, so a missing attachment never costs you a training run.
TIMING_PROBE = False
PROBE_STEPS  = 150

# Hard wall-clock stop. Kaggle kills a committed GPU session at 12 h with no
# warning and no output. We stop ourselves well before that, saving a resumable
# checkpoint, and print the instructions for continuing in the next session.
SESSION_HOURS = 8.0

# Keep this the SAME for all four rungs or you introduce a confound. Single GPU
# is the safe default: DataParallel + gradient checkpointing is a known source
# of subtle breakage, and the guide's own estimate for it is only 1.3-1.5x.
USE_BOTH_GPUS = False

BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
MAX_DUR    = 10.0      # seconds; longer clips blow up memory on a 16 GB card
BATCH      = 8
GRAD_ACCUM = 4         # effective batch 32
LR         = 3e-4
WARMUP     = 500

# ONE step budget for every rung. The experiment is "only the data changes", so
# the optimisation budget must not change with it either. Rungs that converge
# sooner are cut short by EARLY STOPPING, not by a smaller budget, and the
# checkpoint that gets evaluated is always the best-dev-WER one.
#
# The previous per-rung schedule {1:2000, 5:6000, 25:12000, 100:25000} both
# confounded data with budget AND could not fit Kaggle's session limit at the
# top of the ladder. Lower this number if the timing probe says the ladder will
# not fit in your weekly quota -- the small rungs will not notice.
MAX_STEPS           = 6000
EARLY_STOP_PATIENCE = 4          # 4 evals x 500 steps = 2000 steps without gain
EVAL_STEPS = SAVE_STEPS = 500

# Milestone gate, carried over from NB-01 UNCHANGED.
#
# The published 1 h and 5 h runs saved no milestones; the published 25 h run saved
# three. A replicate must inherit its own rung's gate, or the callback set differs
# from the run it is being compared against. The trajectory override that NB-01
# carried here is removed: this notebook never overrides the gate.
MILESTONE_STEPS = (1000, 2000, 3500) if RUNG >= 25 else ()

# Each replicate publishes to its OWN dataset. The suffix is the seed.
DS_NAME = f"ckpt-xlsr-{RUNG}h-s{SEED}"

# =============================================================================
# GUARD -- refuse to start if this is not a replicate of the published run.
#
# Every value below is what the four published ladder runs used. A replicate that
# changes one of them is a new condition, not a second draw, and pooling it as a
# replicate would understate the very variance it was built to measure. This
# fires in milliseconds, before any input is touched.
# =============================================================================
_PUBLISHED = dict(base_model="facebook/wav2vec2-xls-r-300m", max_dur=10.0,
                  batch=8, grad_accum=4, lr=3e-4, warmup=500, max_steps=6000,
                  early_stop_patience=4, eval_steps=500, save_steps=500,
                  use_both_gpus=False)
_THIS = dict(base_model=BASE_MODEL, max_dur=MAX_DUR, batch=BATCH,
             grad_accum=GRAD_ACCUM, lr=LR, warmup=WARMUP, max_steps=MAX_STEPS,
             early_stop_patience=EARLY_STOP_PATIENCE, eval_steps=EVAL_STEPS,
             save_steps=SAVE_STEPS, use_both_gpus=USE_BOTH_GPUS)
_diff = {k: (v, _THIS[k]) for k, v in _PUBLISHED.items() if _THIS[k] != v}
assert not _diff, (
    "This is NOT a replicate of the published ladder. Differences "
    "(published -> this notebook):\n  "
    + "\n  ".join(f"{k}: {a!r} -> {b!r}" for k, (a, b) in _diff.items())
    + "\nRevert them, or stop calling the output a replicate.")

assert RUNG in (1, 5, 25), (
    f"RUNG={RUNG}. The 100 h rung is the one the iso-WER pairs use MILESTONES of, not "
    "its final checkpoint, so a second 100 h final buys the least per GPU-hour. "
    "Replicate 1, 5 and 25 first.")
assert isinstance(SEED, int) and SEED != 1337, (
    f"SEED={SEED}. 1337 is the published ladder's seed and ckpt-xlsr-{RUNG}h already "
    "exists at it. Pick 1338, 1339, ... so the output dataset name is unambiguous. "
    "(Re-running AT 1337 is a legitimate experiment -- it is what produced the paper's "
    "34 pp result -- but publish it under a name that is not the original's.)")
assert DS_NAME != f"ckpt-xlsr-{RUNG}h", "DS_NAME would collide with the published rung"

print(f"REPLICATE | rung {RUNG} h | seed {SEED} | publishes {DS_NAME}")
print("milestones:", MILESTONE_STEPS or "none (matches this rung's published run)")
print(f"config verified identical to the published ladder on {len(_PUBLISHED)} parameters")

# =============================================================================
# PRE-FLIGHT: can this session obtain the BASE MODEL at all?
#
# `Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-xls-r-300m")` is a Hugging
# Face Hub download. It is not on Kaggle's local disk and it is not one of the
# attached datasets. With Internet off, DNS fails and huggingface_hub retries
# with exponential backoff -- config.json five times, adapter_config.json five
# times, config.json five times again. Measured: **840 s before the notebook
# finally raised**, with nothing trained.
#
# This check fires in about five seconds instead, and it runs HERE, before
# transformers has been imported, because HF_HUB_OFFLINE is read at import time.
#
# NOTE: only THIS notebook needs the network. NB-15, NB-16 and NB-17 load models
# from attached Kaggle datasets by local path and make no hub call, so they run
# with Internet off.
# =============================================================================
import os as _os, socket as _socket

def _can_resolve(host="huggingface.co", timeout=5.0):
    try:
        _socket.setdefaulttimeout(timeout)
        _socket.getaddrinfo(host, 443)
        return True
    except OSError:
        return False

ONLINE = _can_resolve()

BASE_CACHED = False
try:
    from huggingface_hub import try_to_load_from_cache
    _hit = try_to_load_from_cache(BASE_MODEL, "config.json")
    BASE_CACHED = isinstance(_hit, str) and _os.path.exists(_hit)
except Exception as _e:
    print(f"  (could not query the HF cache: {type(_e).__name__}: {_e})")

print(f"\nnetwork to huggingface.co : {'YES' if ONLINE else 'NO'}")
print(f"{BASE_MODEL} in local HF cache : {'YES' if BASE_CACHED else 'NO'}")

if not ONLINE and BASE_CACHED:
    # Cached but offline: force offline mode so huggingface_hub does not spend
    # eleven minutes retrying DNS for a file it already has.
    _os.environ["HF_HUB_OFFLINE"] = "1"
    _os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("  -> offline, but the base model is cached. Forcing offline mode so")
    print("     huggingface_hub does not retry DNS for a file it already has.")

assert ONLINE or BASE_CACHED, (
    "\n"
    "+-------------------------------------------------------------------+\n"
    "|  TURN INTERNET ON.  Settings -> Internet -> On  (then re-commit)  |\n"
    "+-------------------------------------------------------------------+\n"
    f"This notebook fine-tunes from {BASE_MODEL}, which is a Hugging Face Hub\n"
    "download of ~1.2 GB. It is not on Kaggle's disk and it is not one of the\n"
    "attached datasets, so with Internet off there is no way to obtain it.\n"
    "\n"
    "Without this check the run would not fail here. It would spend about 840 s\n"
    "in DNS backoff -- fifteen retries across three files -- and raise\n"
    "OSError from the model-loading cell with nothing trained.\n"
    "\n"
    "NB-15, NB-16 and NB-17 are different: they load models from attached\n"
    "datasets by local path and need no network at all.\n"
    "\n"
    "There is no pip install in this notebook, so turning Internet on changes\n"
    "nothing else. It cannot silently re-pin a package version.")

PROV_NET = dict(online=ONLINE, base_model_cached=BASE_CACHED)

## 2 · Boilerplate

In [ ]:
import os
# Must be set BEFORE torch initialises CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" if USE_BOTH_GPUS else "0"
import sys, json, random, time, re, glob, shutil
import numpy as np, pandas as pd, torch
torch.backends.cudnn.benchmark = True

# SEED comes from the CONFIGURATION CELL, not from here.
#
# NB-01 hardcoded `SEED = 1337` at this point. Leaving that line in place would
# silently overwrite the seed set two cells up, and every "replicate" would train
# at 1337 while its dataset name claimed otherwise -- a wrong number that looks
# right. The assertion makes the dependency explicit instead.
assert "SEED" in globals(), "run the configuration cell first -- SEED is set there"
print(f"seeding everything with SEED={SEED} (from the configuration cell)")
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

WORK, TEMP = "/kaggle/working", "/kaggle/temp"
CKPT = f"{WORK}/ckpt"
os.makedirs(f"{TEMP}/scratch", exist_ok=True)

# Used only for dataset-metadata.json. Harmless if it falls back.
KAGGLE_USER = os.environ.get("KAGGLE_USERNAME") or "YOURUSERNAME"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Settings -> Accelerator -> GPU T4 x2"
N_GPU = torch.cuda.device_count()
print(f"visible GPUs: {N_GPU} | {torch.cuda.get_device_name(0)}")
print("T4 has tensor cores, so fp16 training is FASTER here than on a P100.")
if KAGGLE_USER == "YOURUSERNAME":
    print("note: KAGGLE_USERNAME not set; dataset-metadata.json will carry a placeholder id.")

PROV = {"notebook": "NB-14-train-replicate", "rung_hours": RUNG, "seed": SEED,
        "replicate_of": f"ckpt-xlsr-{RUNG}h", "ds_name": DS_NAME,
        "online": PROV_NET["online"], "base_model_cached": PROV_NET["base_model_cached"],
        "base_model": BASE_MODEL, "gpu": torch.cuda.get_device_name(0), "n_gpu": N_GPU,
        "use_both_gpus": USE_BOTH_GPUS, "timing_probe": TIMING_PROBE,
        "max_steps": MAX_STEPS, "session_hours": SESSION_HOURS,
        "batch": BATCH, "grad_accum": GRAD_ACCUM, "lr": LR, "max_dur": MAX_DUR,
        "torch": torch.__version__, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# =============================================================================
# NO PIP, AND WHY -- plus the one thing that pip used to supply.
#
# NB-01's version of this cell ran:
#     !pip install -q transformers==4.44.2 accelerate==1.13.0 librosa soundfile \
#                     jiwer "numpy=={NP_BEFORE}"
# and then `import transformers, accelerate, jiwer`.
#
# With internet OFF that install cannot reach the index. Measured three times in
# this project's own logs: it spends ~202 s timing out, prints two ERROR lines,
# fails, and leaves the preinstalled packages in place. But `jiwer` is NOT
# preinstalled on Kaggle, so the import then raises ModuleNotFoundError -- and
# `compute_metrics` calls `jiwer.wer` at EVERY eval step. The run would die at
# step 500, roughly 18 minutes in.
#
# So WER and CER are implemented here instead, with exactly jiwer's list
# semantics: CORPUS-level, total edits over total reference length, which is what
# `jiwer.wer(list, list)` returns. Verified against jiwer 4.x to machine
# precision in NB-02c. This matters beyond logging, because
# `metric_for_best_model="wer"` means the scorer CHOOSES the published
# checkpoint; a per-utterance mean here would select a different one.
#
# THE ENVIRONMENT CAVEAT, which belongs in the paper and not just here.
# Dropping the pin means this replicate trains on whatever Kaggle currently
# preinstalls, not on the transformers 4.44.2 / accelerate 1.13.0 that the
# published ladder used. The consequence:
#
#   * replicate-vs-replicate comparisons (s1338 vs s1339) are CLEAN -- same
#     environment, same session window, only the run differs. Run all of them
#     close together in time so this stays true.
#   * replicate-vs-published comparisons carry a package-version difference on
#     top of the run difference, and must be reported as such.
#
# Attack-side ASR is unaffected either way: NB-16 attacks every checkpoint in one
# session on one version.
#
# If internet is ever ON, restoring the NB-01 cell verbatim is the better option
# -- but check that pip resolves 4.44.2 against the CURRENT base image before
# trusting it, because a mid-kernel downgrade of an already-imported package is
# how this project's numpy 1.26/2.x hybrid failure happened.
# =============================================================================
import transformers, accelerate, librosa, numpy
print(f"transformers {transformers.__version__} | accelerate {accelerate.__version__}")
print(f"numpy {numpy.__version__} | librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: the published ladder used transformers 4.44.2. This run uses "
          f"{transformers.__version__}.")
    print("  Record it against every number, and keep the replicates in one window.")
PROV["transformers"] = transformers.__version__
PROV["accelerate"] = accelerate.__version__
PROV["numpy"] = numpy.__version__
PROV["librosa"] = librosa.__version__
PROV["env_matches_published_ladder"] = (transformers.__version__ == "4.44.2"
                                        and accelerate.__version__ == "1.13.0")


def _edit(a, b):
    """Levenshtein distance between two token sequences."""
    if not a:
        return len(b)
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(b) + 1):
            cur = d[j]
            d[j] = min(d[j] + 1, d[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return d[len(b)]


def corpus_wer_cer(refs, hyps):
    """jiwer's list semantics: total edits / total reference length, NOT the
    mean of per-utterance rates. Returns (wer, cer)."""
    wn = sum(_edit(r.split(), h.split()) for r, h in zip(refs, hyps))
    wd = sum(len(r.split()) for r in refs)
    cn = sum(_edit(list(r), list(h)) for r, h in zip(refs, hyps))
    cd = sum(len(r) for r in refs)
    return (wn / wd if wd else 1.0), (cn / cd if cd else 1.0)


# Cheap self-test, so a broken scorer is caught here and not by a strange
# checkpoint selection six hours from now.
_w, _c = corpus_wer_cer(["a b c", "d e"], ["a x c", "d e"])
assert abs(_w - 1 / 5) < 1e-12 and _c > 0, (_w, _c)
_w0, _c0 = corpus_wer_cer(["a b"], ["a b"])
assert _w0 == 0.0 and _c0 == 0.0
print("inline corpus WER/CER self-test passed")

## 3 · Load the frozen manifests from NB-00

In [ ]:
# =============================================================================
# Kaggle input resolver  --  paste this in place of the failing SPLITS cell.
#
# Kaggle does not use one mount layout. Depending on the kernel you get:
#     /kaggle/input/<name>                  (older / some kernels)
#     /kaggle/input/datasets/<name>         (your own datasets, current)
#     /kaggle/input/competitions/<name>     (competition data)
# A flat glob("/kaggle/input/*") only sees the wrapper dirs, which is why
# NB-01 reported: Visible inputs: ['competitions', 'datasets']
# =============================================================================
import os, re, glob

def list_inputs():
    out = []
    for pat in ["/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"]:
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))


def find_input(pattern, what="input", required=True):
    """Find a mounted input directory by regex on its BASENAME, at any depth."""
    rx = re.compile(pattern, re.I)
    hits = [p for p in list_inputs() if rx.search(os.path.basename(p))]
    if not hits:
        if not required:
            return None
        print("Directories visible under /kaggle/input:")
        for p in list_inputs():
            print("   ", p)
        raise AssertionError(f"Could not find {what}  (regex: {pattern})")
    # deepest match wins: /kaggle/input/datasets/bn-asr-splits over /kaggle/input/datasets
    best = max(hits, key=lambda p: (p.count("/"), len(p)))
    if len(hits) > 1:
        print(f"  ({what}: {len(hits)} matches, using deepest) {hits}")
    return best


SPLITS = find_input(r"splits", "your NB-00 output dataset 'bn-asr-splits'")
print("splits dataset:", SPLITS)
print(sorted(os.listdir(SPLITS)))

train_df = pd.read_csv(f"{SPLITS}/rung_{RUNG}h.csv")
dev_df   = pd.read_csv(f"{SPLITS}/dev.csv").sample(500, random_state=SEED)
vocab    = json.load(open(f"{SPLITS}/vocab.json"))

# -----------------------------------------------------------------------------
# The manifests hold ABSOLUTE paths into the competition mount. If that mount
# moved between notebooks, every path breaks -- and it would fail later, inside
# the dataloader, as thousands of silent zero-audio samples rather than an
# error. Verify now and remap the prefix if needed.
# -----------------------------------------------------------------------------
probe = str(train_df.iloc[0]["path"])
if not os.path.exists(probe):
    print(f"\nmanifest paths are stale:\n  {probe}")
    new_root = find_input(r"bengali.*speech|bengaliai", "the bengaliai-speech competition data")
    m = re.match(r"(.*?bengaliai[^/]*)/", probe)
    old_root = m.group(1) if m else None
    assert old_root, f"cannot parse the corpus root out of {probe}"
    print(f"remapping {old_root}  ->  {new_root}")
    for d in (train_df, dev_df):
        d["path"] = d["path"].str.replace(old_root, new_root, regex=False)
    probe = str(train_df.iloc[0]["path"])
    assert os.path.exists(probe), f"remap failed, still missing: {probe}"
    print("remap OK")
else:
    print("\nmanifest audio paths resolve correctly")

train_df = train_df[train_df.duration <= MAX_DUR].reset_index(drop=True)
dev_df   = dev_df[dev_df.duration <= MAX_DUR].reset_index(drop=True)

print(f"\nrung {RUNG}h -> {len(train_df)} train utts | {train_df.duration.sum()/3600:.2f} h")
print(f"dev          -> {len(dev_df)} utts")
print(f"vocab size   -> {len(vocab)}")


## 4 · Tokenizer, feature extractor, processor

In [ ]:
from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                          Wav2Vec2Processor, Wav2Vec2ForCTC)

json.dump(vocab, open(f"{WORK}/vocab.json", "w"), ensure_ascii=False)
tokenizer = Wav2Vec2CTCTokenizer(f"{WORK}/vocab.json", unk_token="<unk>",
                                 pad_token="<pad>", word_delimiter_token="|")
fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0,
                              do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=fe, tokenizer=tokenizer)
processor.save_pretrained(f"{WORK}/processor")

print("pad/blank id:", tokenizer.pad_token_id, "| vocab:", len(tokenizer))
print("NOTE: do_normalize=True here -> NB-04's differentiable front-end must match this.")

## 5 · Dataset — decode audio on the fly, never cache it to /kaggle/working

In [ ]:
import librosa
from torch.utils.data import Dataset

class BnASR(Dataset):
    """Decodes audio on the fly.

    A failed decode is NEVER silently replaced by silence. The original version
    swallowed every exception and returned a second of zeros, which meant a
    broken audio path produced a full-length training run on nothing.
    """
    n_fail   = 0
    MAX_FAIL = 20          # per dataloader worker; see the note below

    def __init__(self, df, processor, max_dur=MAX_DUR):
        self.df = df.reset_index(drop=True); self.p = processor; self.max_dur = max_dur

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:
            a, _ = librosa.load(r["path"], sr=16000, mono=True, duration=self.max_dur)
            if a.size < 400:
                raise ValueError(f"decoded only {a.size} samples")
        except Exception as e:
            BnASR.n_fail += 1
            print(f"AUDIO LOAD FAILED ({BnASR.n_fail}): {r['path']} -> "
                  f"{type(e).__name__}: {e}", flush=True)
            # dataloader_num_workers>0 forks this class, so the counter is
            # per-worker and the real tolerance is MAX_FAIL x n_workers.
            # It is a tripwire, not an accountant.
            assert BnASR.n_fail < BnASR.MAX_FAIL, (
                "too many audio load failures -- aborting rather than training on silence")
            a = np.zeros(16000, dtype=np.float32)
        a = np.asarray(a, dtype=np.float32)
        vals = self.p.feature_extractor(a, sampling_rate=16000).input_values[0]
        # processor.as_target_processor() is deprecated; the tokenizer directly
        # is identical in behaviour and survives a transformers bump.
        labels = self.p.tokenizer(str(r["text"])).input_ids
        return {"input_values": vals, "labels": labels}


from dataclasses import dataclass
@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        inp = [{"input_values": f["input_values"]} for f in features]
        lab = [{"input_ids":   f["labels"]}        for f in features]
        batch = self.processor.feature_extractor.pad(inp, padding=True, return_tensors="pt")
        lb    = self.processor.tokenizer.pad(lab, padding=True, return_tensors="pt")
        # -100 tells the CTC loss to ignore padding positions
        batch["labels"] = lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        return batch

train_ds, dev_ds = BnASR(train_df, processor), BnASR(dev_df, processor)
collator = Collator(processor)

_b = collator([train_ds[0], train_ds[1]])
print("sanity:", {k: tuple(v.shape) for k, v in _b.items()})
assert _b["input_values"].abs().sum() > 0, \
    "the first two training clips decoded to pure silence -- check AUDIO_ROOT"
assert (_b["labels"] != -100).any(), "no label tokens survived collation"
print("first two clips decoded to real audio and real labels")

## 6 · Model

In [ ]:
model, load_info = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.05, hidden_dropout=0.05, feat_proj_dropout=0.05,
    mask_time_prob=0.05, layerdrop=0.05,
    ctc_loss_reduction="mean",
    # A single inf CTC loss (target longer than the frame sequence) would
    # otherwise poison the batch and NaN the weights under fp16.
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    output_loading_info=True,           # <- lets us VERIFY what actually loaded
)

# ---------------------------------------------------------------------------
# torch >= 2.1 renamed weight-norm parameters:
#     weight_g / weight_v  ->  parametrizations.weight.original0 / original1
# NB-04's log showed the checkpoint's pos_conv weights going UNUSED while fresh
# ones were initialised. If that happens here the positional convolution starts
# RANDOM instead of pretrained -- and the 1h rung, where data is scarcest, pays
# the highest price. That would corrupt RQ1 exactly where it matters most.
# ---------------------------------------------------------------------------
pos_missing = [k for k in load_info.get("missing_keys", [])    if "pos_conv" in k]
pos_unused  = [k for k in load_info.get("unexpected_keys", []) if "pos_conv" in k]
print("pos_conv missing (randomly init):", pos_missing)
print("pos_conv unused  (in ckpt, ignored):", pos_unused)

if pos_missing:
    print("\n*** REPAIRING positional conv embedding ***")
    from huggingface_hub import hf_hub_download
    sd = None
    for fn in ["model.safetensors", "pytorch_model.bin"]:
        try:
            p = hf_hub_download(BASE_MODEL, fn)
            if fn.endswith(".safetensors"):
                import safetensors.torch as sft; sd = sft.load_file(p)
            else:
                sd = torch.load(p, map_location="cpu", weights_only=True)
            print("  loaded raw checkpoint:", fn); break
        except Exception as e:
            print(f"  {fn}: {type(e).__name__}")

    conv = model.wav2vec2.encoder.pos_conv_embed.conv
    fixed = False
    if sd is not None and hasattr(conv, "parametrizations"):
        base = "wav2vec2.encoder.pos_conv_embed.conv."
        g, v = sd.get(base + "weight_g"), sd.get(base + "weight_v")
        if g is not None and v is not None:
            o0 = conv.parametrizations.weight.original0
            o1 = conv.parametrizations.weight.original1
            if o0.shape == g.shape and o1.shape == v.shape:
                with torch.no_grad():
                    o0.copy_(g.to(o0.dtype)); o1.copy_(v.to(o1.dtype))
                fixed = True
                print("  remapped weight_g/weight_v -> parametrizations original0/1  OK")
            else:
                print(f"  SHAPE MISMATCH g{tuple(g.shape)} vs o0{tuple(o0.shape)} | "
                      f"v{tuple(v.shape)} vs o1{tuple(o1.shape)}")
    if not fixed:
        print("  COULD NOT REPAIR. Record as a limitation and treat the 1h rung with")
        print("  caution -- it is the rung most sensitive to lost pretraining.")
    PROV["pos_conv_repaired"] = fixed
else:
    print("\npositional conv loaded cleanly from the checkpoint.")
    PROV["pos_conv_repaired"] = None

model.freeze_feature_encoder()     # standard for w2v2 fine-tuning; saves GPU time

n_all = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nparameters: {n_all/1e6:.1f} M total | {n_tr/1e6:.1f} M trainable")


## 7 · Trainer, with checkpoint-and-resume

Three things here are load-bearing:

* **Resume looks in an attached *input* dataset, not just `/kaggle/working`.** A committed run
  always starts with an empty `working`, so the old "is there a checkpoint in working?" test
  could never fire across sessions — the resume path was dead code.
* **`save_total_limit=2`, not 1.** HF keeps `{best, latest}`. The *latest* is what a resume
  needs; the *best* is what gets published. With a limit of 1 the resume target could be
  deleted out from under the next session.
* **Length-bucketed batches.** The mean clip is 4.4 s but a batch pads to its longest member,
  so random batching spends roughly 40% of its FLOPs on zeros. We hand the sampler the
  durations we already have in the manifest, rather than letting HF decode a million files to
  rediscover them.

In [ ]:
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          EarlyStoppingCallback)
try:
    from transformers.trainer_pt_utils import LengthGroupedSampler
except ImportError:                      # moved or renamed in some versions
    LengthGroupedSampler = None

def compute_metrics(pred):
    ids = np.argmax(pred.predictions, axis=-1)
    lab = pred.label_ids.copy()
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyp = processor.batch_decode(ids)
    ref = processor.batch_decode(lab, group_tokens=False)
    pairs = [(r, h) for r, h in zip(ref, hyp) if r.strip()]
    if not pairs: return {"wer": 1.0, "cer": 1.0}
    ref_s = [p[0] for p in pairs]; hyp_s = [p[1] for p in pairs]
    # jiwer is not available (see the no-pip cell); corpus_wer_cer is its
    # list-semantics equivalent, defined there.
    _w, _c = corpus_wer_cer(ref_s, hyp_s)
    return {"wer": _w, "cer": _c}

# ---------------------------------------------------------------------------
# Resume. Look in an ATTACHED INPUT first.
#
# /kaggle/working starts EMPTY in every committed run, so the original
# "is there a checkpoint in /kaggle/working/ckpt?" test could never fire across
# sessions -- the resume path was dead code. To continue a rung: publish the
# previous run's output as ckpt-xlsr-{RUNG}h, attach it as an input, re-commit.
# ---------------------------------------------------------------------------
def _ckpts(d):
    if not os.path.isdir(d): return []
    return sorted((x for x in os.listdir(d) if re.fullmatch(r"checkpoint-\d+", x)),
                  key=lambda s: int(s.split("-")[1]))

os.makedirs(CKPT, exist_ok=True)
if not _ckpts(CKPT):
    # `ckpt-xlsr-5h$` cannot match `ckpt-xlsr-25h`; a bare "5h" substring test could.
    prev = [p for p in glob.glob("/kaggle/input/*")
            if re.search(rf"{re.escape(DS_NAME)}$", os.path.basename(p))]
    for p in prev:
        cs = _ckpts(f"{p}/ckpt")
        if cs:
            print(f"copying {cs[-1]} from {os.path.basename(p)} (a few minutes)...", flush=True)
            shutil.copytree(f"{p}/ckpt/{cs[-1]}", f"{CKPT}/{cs[-1]}")
            break

resume, START_STEP = None, 0
_have = _ckpts(CKPT)
if _have:
    resume = f"{CKPT}/{_have[-1]}"
    START_STEP = json.load(open(f"{resume}/trainer_state.json"))["global_step"]
    print(f"RESUMING from {resume} -- step {START_STEP} of {MAX_STEPS}")
else:
    print("fresh start (no checkpoint in /kaggle/working and none in an attached input)")


class TimeBudget(TrainerCallback):
    """Stop and SAVE before Kaggle's 12 h guillotine, so a long rung can be
    resumed next session instead of vanishing with the container."""
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
        self.hit = False
    def on_step_end(self, args, state, control, **kw):
        if time.time() >= self.deadline:
            self.hit = True
            print(f"\n*** SESSION BUDGET ({SESSION_HOURS} h) reached at step "
                  f"{state.global_step}/{MAX_STEPS}. Saving and stopping. ***", flush=True)
            control.should_save = True
            control.should_training_stop = True
        return control


class NanGuard(TrainerCallback):
    """fp16 + CTC can diverge to NaN. Dying at step 300 beats producing a model
    of NaNs at step 6000."""
    bad = 0
    def on_log(self, args, state, control, logs=None, **kw):
        l = (logs or {}).get("loss")
        if l is None: return control
        NanGuard.bad = NanGuard.bad + 1 if (np.isnan(l) or np.isinf(l)) else 0
        if NanGuard.bad >= 5:
            print("\n*** loss NaN/inf for 5 consecutive logs -- stopping. "
                  "Lower LR or disable fp16. ***", flush=True)
            control.should_training_stop = True
        return control


class Milestones(TrainerCallback):
    """Weights-only snapshots for NB-02's iso-WER arm. ~1.3 GB each instead of
    ~3.8 GB, because the optimiser state is never needed again.

    Only fires for RUNG >= 25, matching the published gate.

    NEVER raises. NB-01 took `model` from the keyword argument and called
    `.save_pretrained` on it directly; if a future transformers stops passing
    `model=` to `on_save`, that is an AttributeError on None, thrown from inside
    the training loop, three and a half hours in, destroying a finished run over
    an optional 1.3 GB side file. The final checkpoint is what the variance
    measurement needs; a missing milestone is a note, not a failure.
    """
    def on_save(self, args, state, control, model=None, **kw):
        if state.global_step not in MILESTONE_STEPS:
            return control
        try:
            m = model if model is not None else kw.get("model")
            if m is None:
                # Last resort: the trainer binds itself at module scope below.
                m = getattr(globals().get("trainer", None), "model", None)
            if m is None:
                print(f"milestone at step {state.global_step} SKIPPED: this "
                      "transformers does not hand the model to on_save. The run "
                      "is unaffected; /final is still written.")
                return control
            free = shutil.disk_usage(WORK).free / 1e9
            if free < 4.0:
                print(f"skipping milestone at step {state.global_step}: "
                      f"only {free:.1f} GB free")
                return control
            d = f"{WORK}/milestones/step-{state.global_step}"
            (m.module if hasattr(m, "module") else m).save_pretrained(
                d, safe_serialization=True)
            processor.save_pretrained(d)
            print(f"milestone saved: {d}  ({free:.1f} GB free before the write)")
        except Exception as e:
            print(f"milestone at step {state.global_step} FAILED "
                  f"({type(e).__name__}: {e}) -- continuing. Milestones are "
                  "optional here; /final is what this notebook exists to produce.")
        return control


# =============================================================================
# TrainingArguments, built as a dict and then FILTERED against the installed
# signature.
#
# Why: `transformers` 5.x removed `group_by_length`, and passing it raises
#     TypeError: __init__() got an unexpected keyword argument 'group_by_length'
# about 80 s into the run. The published ladder trained on 4.44.2, where it was
# still accepted; Kaggle now ships 5.16.1.
#
# A blanket try/except or a bare `**{k: v for k in sig}` would be worse than the
# crash. Silently dropping `fp16` would train in fp32 -- different speed AND a
# different nondeterminism profile, which is the quantity this whole experiment
# measures. Silently dropping `max_steps` or `seed` would wreck the comparison
# outright.
#
# So: only the arguments that are genuinely inert here may be dropped, and
# anything else missing from the signature is a hard stop with its name printed.
# =============================================================================
_WANT = dict(
    output_dir=CKPT,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    # HF multiplies batch by device count. Halve accumulation on 2 GPUs so the
    # effective batch stays 32 and the rungs remain directly comparable.
    gradient_accumulation_steps=max(1, GRAD_ACCUM // max(1, N_GPU)),
    gradient_checkpointing=True,
    # use_reentrant=False: the reentrant implementation silently produces no
    # gradients when no input requires grad -- which is exactly our case, since
    # the feature encoder is frozen.
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    learning_rate=LR, warmup_steps=WARMUP,
    max_steps=PROBE_STEPS if TIMING_PROBE else MAX_STEPS,
    eval_strategy="no" if TIMING_PROBE else "steps", eval_steps=EVAL_STEPS,
    save_strategy="no" if TIMING_PROBE else "steps", save_steps=SAVE_STEPS,
    # 2, not 1: HF keeps {best, latest}. The latest is what a resume needs, the
    # best is what gets published. With save_total_limit=1 the resume target
    # could be deleted out from under the next session.
    save_total_limit=2,
    load_best_model_at_end=not TIMING_PROBE,
    metric_for_best_model="wer", greater_is_better=False,
    logging_steps=25, report_to="none",
    dataloader_num_workers=2,
    # Inert: we install our own length-grouped sampler below, so HF's own
    # grouping is off either way. Removed in transformers 5.x.
    group_by_length=False,
    seed=SEED,
)

# Arguments it is safe to lose, with the reason each one is inert:
#   group_by_length -- False anyway; LadderTrainer supplies the sampler
#   report_to       -- logging destination only; "none" is also the default here
_SAFE_TO_DROP = {"group_by_length", "report_to"}

import inspect
_sig = set(inspect.signature(TrainingArguments.__init__).parameters)
if "kwargs" in _sig or "kw" in _sig:
    # A **kwargs signature accepts everything and tells us nothing, so the
    # filter cannot help; pass it all through and let it raise if it must.
    _drop = []
else:
    _drop = sorted(k for k in _WANT if k not in _sig)
_unsafe = [k for k in _drop if k not in _SAFE_TO_DROP]
assert not _unsafe, (
    f"transformers {transformers.__version__} does not accept {_unsafe}, and these "
    "are not safe to drop -- each one changes what gets trained. Pin transformers "
    "to 4.44.2 with internet ON, or find the replacement argument name and set it "
    "explicitly. Do NOT just delete them.")
for _k in _drop:
    _WANT.pop(_k)
    print(f"dropped inert argument '{_k}' (not in transformers "
          f"{transformers.__version__}); it was {'False' if _k == 'group_by_length' else 'cosmetic'}")

args = TrainingArguments(**_WANT)
print(f"TrainingArguments built with {len(_WANT)} arguments on transformers "
      f"{transformers.__version__}")

# The settings this experiment cannot tolerate losing, re-read off the object
# that was actually constructed rather than off the dict we hoped to pass.
for _k, _v in (("max_steps", _WANT["max_steps"]), ("seed", SEED), ("fp16", True),
               ("learning_rate", LR), ("save_total_limit", 2),
               ("gradient_accumulation_steps", _WANT["gradient_accumulation_steps"])):
    _got = getattr(args, _k, "<<absent>>")
    assert _got == _v, (f"TrainingArguments.{_k} is {_got!r}, expected {_v!r}. "
                        "The replicate would not match the published ladder.")
print("verified on the constructed object: max_steps, seed, fp16, lr, "
      "save_total_limit, grad_accum")
PROV["train_args_dropped"] = _drop

# ---------------------------------------------------------------------------
# Length-bucketed batches.
#
# Mean clip is 4.4 s but a batch pads to its longest member, so random batching
# burns roughly 40% of its FLOPs on zeros. Grouping by length removes most of
# that. HF's own group_by_length=True would have to DECODE every file to
# discover lengths; we already have them in the manifest, so we hand them over.
# Applied identically to every rung -- not a confound, but record it anyway.
# ---------------------------------------------------------------------------
TRAIN_LENGTHS = (np.minimum(train_df["duration"].values, MAX_DUR) * 16000).astype(int).tolist()

class LadderTrainer(Trainer):
    def _get_train_sampler(self, *a, **k):
        # Fail soft: a slower run beats a run that will not start.
        if LengthGroupedSampler is None:
            return super()._get_train_sampler(*a, **k)
        try:
            g = torch.Generator(); g.manual_seed(SEED)
            return LengthGroupedSampler(
                batch_size=self.args.train_batch_size * self.args.gradient_accumulation_steps,
                lengths=TRAIN_LENGTHS, generator=g)
        except Exception as e:
            print(f"length grouping unavailable ({type(e).__name__}: {e}) -- "
                  "falling back to the default sampler", flush=True)
            return super()._get_train_sampler(*a, **k)

TB  = TimeBudget(SESSION_HOURS)
cbs = [TB, NanGuard()]
if not TIMING_PROBE:
    cbs.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE))
    if MILESTONE_STEPS: cbs.append(Milestones())

# `Trainer(tokenizer=...)` was deprecated in transformers 4.46 and REMOVED in
# 5.x, replaced by `processing_class`. Without this the run would die here --
# the next TypeError after group_by_length, another 80 s in.
_tsig = set(inspect.signature(Trainer.__init__).parameters)
_proc_kw = "processing_class" if "processing_class" in _tsig else "tokenizer"
assert _proc_kw in _tsig, (
    f"transformers {transformers.__version__} Trainer accepts neither "
    "`processing_class` nor `tokenizer`. Check the signature before running.")
print(f"passing the feature extractor as Trainer({_proc_kw}=...)")

trainer = LadderTrainer(model=model, args=args, data_collator=collator,
                        train_dataset=train_ds, eval_dataset=dev_ds,
                        compute_metrics=compute_metrics,
                        callbacks=cbs,
                        **{_proc_kw: processor.feature_extractor})

print(f"\nmode           : {'TIMING PROBE' if TIMING_PROBE else 'FULL TRAINING'}")
print(f"max_steps      : {args.max_steps}")
print(f"effective batch: {args.train_batch_size * args.gradient_accumulation_steps}")
print(f"resume from    : {resume}")
print(f"milestones     : {MILESTONE_STEPS or 'none (small rung)'}")
print(f"free disk      : {shutil.disk_usage(WORK).free/1e9:.1f} GB of 20 GB")
# ---------------------------------------------------------------------------
# EPOCHS. A fixed step budget means each rung makes a very different number of
# passes over its own data. That is inherent to holding COMPUTE constant while
# varying data, and it is the honest framing -- but record it, because "the 1 h
# model saw its 809 clips ~237 times" is a limitation a reviewer will ask about
# and you want the number ready rather than estimated on the spot.
# ---------------------------------------------------------------------------
eff_batch = args.train_batch_size * args.gradient_accumulation_steps
EPOCHS_AT_CAP = MAX_STEPS * eff_batch / max(1, len(train_df))
print(f"\nrung {RUNG}h: {len(train_df)} utts, effective batch {eff_batch}")
print(f"  {MAX_STEPS} steps would be {EPOCHS_AT_CAP:.1f} epochs over this rung")
print("  early stopping should cut the small rungs well short of that.")
PROV["epochs_at_cap"] = round(EPOCHS_AT_CAP, 1)
PROV["effective_batch"] = eff_batch
PROV["actual_train_hours"] = round(float(train_df.duration.sum()/3600), 3)


## 8 · Train

> **On intermediate checkpoints.** Keeping every 500-step checkpoint is not possible — each one
> is ~3.8 GB against a 20 GB cap, and 12 of them would overflow it. Instead, two cheaper things
> give NB-02 what it needs for the iso-WER analysis:
>
> * `log_history.json` — the full dev-WER-vs-step curve, written in Section 9. Free.
> * `MILESTONE_STEPS` — weights-only snapshots (~1.3 GB each, no optimiser state), saved only
>   for the 25 h and 100 h rungs. Those are the only rungs that can play the "big model stopped
>   early, WER-matched to a small model" role that iso-WER actually needs.

In [ ]:
t0 = time.time()
result = trainer.train(resume_from_checkpoint=resume)
elapsed = time.time() - t0

steps_done   = max(1, trainer.state.global_step - START_STEP)
sec_per_step = elapsed / steps_done

print(f"\nwall-clock this session : {elapsed/3600:.2f} h for {steps_done} steps")
print(f"measured throughput     : {sec_per_step:.2f} s/step "
      f"({3600/sec_per_step:.0f} steps/hour)")

if TIMING_PROBE:
    h = MAX_STEPS * sec_per_step / 3600
    print("\n" + "=" * 68)
    print("  PROJECTED BUDGET  (every rung shares MAX_STEPS, so every rung")
    print("  costs about the same wall-clock regardless of its audio hours)")
    print("=" * 68)
    print(f"  {MAX_STEPS} steps           -> {h:5.1f} h per rung")
    print(f"  four rungs            -> {4*h:5.1f} h for the whole ladder")
    print(f"  sessions per rung     -> {int(np.ceil(h/SESSION_HOURS))} "
          f"(at SESSION_HOURS={SESSION_HOURS})")
    print(f"  Kaggle weekly GPU quota is ~30 h.")
    if 4 * h > 28:
        need = int(28 / (4 * sec_per_step / 3600))
        print(f"\n  THIS DOES NOT FIT. Lower MAX_STEPS to about {need} "
              f"({4*need*sec_per_step/3600:.1f} h total).")
        print("  Early stopping means the 1 h and 5 h rungs will not notice.")
    else:
        print("\n  This fits. Set TIMING_PROBE = False and re-commit to train for real.")
    print("=" * 68)
else:
    print(result.metrics)

## 9 · Final evaluation and save

In [ ]:
def finalise():
    metrics = trainer.evaluate()
    print(json.dumps(metrics, indent=2))

    # The full dev-WER-vs-step curve. Costs nothing, and NB-02 needs it to choose
    # iso-WER operating points without retraining anything.
    json.dump(trainer.state.log_history, open(f"{WORK}/log_history.json", "w"), indent=1)

    stopped_by = ("session_budget" if TB.hit else
                  "max_steps" if trainer.state.global_step >= MAX_STEPS else "early_stopping")

    PROV.update(rung_utts=int(len(train_df)),
                rung_hours_actual=round(float(train_df.duration.sum()/3600), 2),
                steps_completed=int(trainer.state.global_step),
                sec_per_step=round(sec_per_step, 2),
                stopped_by=stopped_by,
                eval_wer=metrics.get("eval_wer"), eval_cer=metrics.get("eval_cer"),
                best_checkpoint=trainer.state.best_model_checkpoint,
                milestones=[s for s in MILESTONE_STEPS
                            if os.path.isdir(f"{WORK}/milestones/step-{s}")])

    if TB.hit:
        print("\n" + "=" * 70)
        print(f"  RUNG {RUNG}h IS NOT FINISHED -- stopped at step "
              f"{trainer.state.global_step}/{MAX_STEPS} by the session budget.")
        print("  Nothing is lost. To continue:")
        print(f"    1. Output panel -> Create Dataset (or New Version) -> {DS_NAME}")
        print(f"    2. + Add Input -> Your Datasets -> {DS_NAME}   (this notebook)")
        print( "    3. Re-commit. It resumes from the checkpoint automatically.")
        print(f"  Estimated remaining: "
              f"{(MAX_STEPS - trainer.state.global_step) * sec_per_step / 3600:.1f} h")
        print("=" * 70)
    else:
        trainer.save_model(f"{WORK}/final")
        processor.save_pretrained(f"{WORK}/final")
        print(f"\nRUNG {RUNG}h COMPLETE ({stopped_by}).")
        print(f"  dev WER {metrics.get('eval_wer'):.4f} | dev CER {metrics.get('eval_cer'):.4f}")
        print(f"  final model -> {WORK}/final")
        print(f"  publish the output as dataset  {DS_NAME}  for NB-02 and NB-05.")

    json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
    json.dump({"title": DS_NAME, "id": f"{KAGGLE_USER}/{DS_NAME}",
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
    print(json.dumps(PROV, indent=2))


# Do NOT `raise SystemExit` for the probe: on a Kaggle commit that marks the whole
# run as failed, which is exactly the wrong signal for a successful probe.
if TIMING_PROBE:
    print("TIMING PROBE -- skipping final evaluation and save.")
    print("Set TIMING_PROBE = False and re-commit to train for real.")
else:
    finalise()
# ---------------------------------------------------------------------------
# DID THIS RUNG CONVERGE, OR DID THE CAP CUT IT OFF?
#
# At 6000 steps the 100 h rung makes only ~2.4 passes over its data. If its dev
# WER is still falling when the cap hits, it is UNDERTRAINED relative to the
# small rungs -- which biases RQ1 toward the null, understating the data
# effect. That is the conservative direction, but it must be reported, not
# discovered by a reviewer.
# ---------------------------------------------------------------------------
hist = [h for h in trainer.state.log_history if "eval_wer" in h]
if len(hist) >= 3:
    steps = [h["step"] for h in hist]
    wers  = [h["eval_wer"] for h in hist]
    best_i = int(np.argmin(wers))
    tail = wers[-3:]
    still_falling = (tail[0] - tail[-1]) > 0.005      # >0.5 WER points over last 3 evals

    print("\ndev WER trajectory:")
    for st, w in zip(steps[-6:], wers[-6:]):
        print(f"   step {st:>6}  WER {w:.4f}")
    print(f"\nbest WER {wers[best_i]:.4f} at step {steps[best_i]} "
          f"(of {steps[-1]} run, cap {MAX_STEPS})")

    if still_falling and steps[-1] >= MAX_STEPS * 0.95:
        print("\n*** STILL IMPROVING AT THE CAP -- this rung is undertrained. ***")
        print("    Either raise MAX_STEPS for ALL rungs and retrain, or report it")
        print("    explicitly as a conservative bias in RQ1.")
    elif best_i < len(wers) - 3:
        print("\nConverged: best checkpoint is well before the end, early stopping worked.")
    else:
        print("\nPlateaued near the cap.")

    PROV["best_wer_step"]     = int(steps[best_i])
    PROV["final_step"]        = int(steps[-1])
    PROV["still_falling_at_cap"] = bool(still_falling and steps[-1] >= MAX_STEPS * 0.95)
    PROV["wer_trajectory"]    = [[int(a), round(float(b), 4)] for a, b in zip(steps, wers)]
else:
    print("\ntoo few eval points for a convergence diagnostic")


In [ ]:
# Free disk before the session ends.
#
#   run finished   -> /final already holds the best-dev-WER weights, so older
#                     checkpoints are redundant. The newest one is kept anyway,
#                     in case the ladder turns out non-monotonic and you want to
#                     extend this rung rather than restart it.
#   run unfinished -> the newest checkpoint IS the resume point. Never delete it.
if not TIMING_PROBE:
    keep = (sorted([d for d in os.listdir(CKPT) if re.fullmatch(r"checkpoint-\d+", d)],
                   key=lambda d: int(d.split("-")[1]))
            if os.path.isdir(CKPT) else [])
    for d in keep[:-1]:
        shutil.rmtree(os.path.join(CKPT, d), ignore_errors=True)
    if keep:
        print(f"kept {keep[-1]}"
              + (" (RESUME POINT -- do not delete)" if TB.hit else " (spare; /final is the model)")
              + f"; removed {len(keep)-1} older checkpoint(s)")

print("working dir size:",
      round(sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(WORK) for f in fs)/1e9, 2), "GB / 20 GB")

---

## Gate before moving on

Record every replicate. `_provenance.json` carries all four columns.

| rung | seed | transformers | dev WER | best step | stopped_by | dataset |
|---|---|---|---|---|---|---|
| 1 h  | 1337 | 4.44.2 | 0.6762 | 2500 | early_stopping | `ckpt-xlsr-1h` (published) |
| 1 h  | 1337 | 4.44.2 | 0.6810 | 5500 | max_steps | `ckpt-xlsr-1h-traj` (the paper's run 2) |
| 1 h  | 1338 | | | | | `ckpt-xlsr-1h-s1338` |
| 1 h  | 1339 | | | | | `ckpt-xlsr-1h-s1339` |
| 5 h  | 1337 | 4.44.2 | 0.4739 | | | `ckpt-xlsr-5h` (published) |
| 5 h  | 1338 | | | | | `ckpt-xlsr-5h-s1338` |
| 5 h  | 1339 | | | | | `ckpt-xlsr-5h-s1339` |
| 25 h | 1337 | 4.44.2 | 0.3484 | | | `ckpt-xlsr-25h` (published) |
| 25 h | 1338 | | | | | `ckpt-xlsr-25h-s1338` |
| 25 h | 1339 | | | | | `ckpt-xlsr-25h-s1339` |

Fill in the `transformers` column from each run's own log line. If Kaggle moves mid-way through
and the six replicates do not all share one version, say so — and set `REPLICATE_ENV` in NB-17
accordingly.

**There is no gate on dev WER here, and that is deliberate.** A replicate that lands further
from its published sibling than you expected is the measurement, not a failure. The only
genuine failures are `stopped_by == "session_budget"` (unfinished — publish, attach back,
re-commit) and a NaN-loss abort.

Do **not** discard a replicate for being an outlier. Selecting replicates on their outcome is
precisely the bias the variance estimate exists to expose.

### Then

1. **NB-15** — clean WER on the 100-utterance attack set for every new checkpoint. ~15 min
   total. This is the ANCOVA covariate; without it a replicate cannot enter the pooled fit.
2. **NB-16** — ε₅₀ at n=100 on every new checkpoint, in Task B's exact attack configuration.
   ~4.6 GPU-h each.
3. **NB-17** — variance components, pooled σ, run-clustered ANCOVA, power curve. CPU, minutes.
